# S23 — Autoencoders and VAEs

**Week 12 · Module 4**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s23_autoencoders_and_vaes.ipynb)

Every cell below is a worked example from the [S23 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s23/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s23.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s23.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
print("environment ready")

## The bottleneck forces structure


*Expected output starts with:* `Linear autoencoder (2 -> 1 -> 2):`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Synthetic 2D data: a noisy arc (a 1D curve embedded in 2D)
n = 2000
t = torch.rand(n, 1) * 3.14159
x = torch.cat([torch.cos(t), torch.sin(t)], dim=1) + 0.05 * torch.randn(n, 2)

linear_ae = nn.Sequential(nn.Linear(2, 1), nn.Linear(1, 2))
nonlinear_ae = nn.Sequential(
    nn.Linear(2, 32), nn.Tanh(), nn.Linear(32, 1),   # encoder -> 1D code
    nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 2),   # decoder -> 2D
)

def train(model, steps=2000):
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    for step in range(steps):
        loss = ((model(x) - x) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 500 == 0 or step == steps - 1:
            print(f"  step {step:4d}  reconstruction MSE {loss.item():.4f}")
    return loss.item()

print("Linear autoencoder (2 -> 1 -> 2):")
lin = train(linear_ae)
print("Nonlinear autoencoder (2 -> 32 -> 1 -> 32 -> 2):")
non = train(nonlinear_ae)
print(f"final MSE: linear {lin:.4f}  nonlinear {non:.4f}")
print(f"data noise floor (0.05^2 per coord): {0.05**2:.4f}")

## The reparameterization trick


*Expected output starts with:* `sample():  z = [2.0409960746765137, -1.293428897857666]`


In [ ]:
import torch
from torch.distributions import Normal

torch.manual_seed(0)

mu = torch.tensor([0.5, -1.0], requires_grad=True)
log_sigma = torch.tensor([0.0, 0.0], requires_grad=True)
dist = Normal(mu, log_sigma.exp())

# Naive sampling: draw z ~ N(mu, sigma^2) directly
z_naive = dist.sample()
print(f"sample():  z = {z_naive.tolist()}")
print(f"sample():  requires_grad = {z_naive.requires_grad}, grad_fn = {z_naive.grad_fn}")
try:
    (z_naive ** 2).sum().backward()
except RuntimeError as e:
    print(f"sample():  backward() fails: {e}")

# Reparameterized sampling: z = mu + sigma * eps, eps ~ N(0, 1)
z_rep = dist.rsample()   # same as mu + log_sigma.exp() * torch.randn(2)
print(f"rsample(): z = {[round(v, 4) for v in z_rep.tolist()]}")
print(f"rsample(): requires_grad = {z_rep.requires_grad}")
(z_rep ** 2).sum().backward()
print(f"rsample(): mu.grad        = {[round(v, 4) for v in mu.grad.tolist()]}")
print(f"rsample(): log_sigma.grad = {[round(v, 4) for v in log_sigma.grad.tolist()]}")

# Check against the closed form: for loss z^2 with z = mu + sigma*eps,
# dL/dmu = 2z and dL/dlog_sigma = 2z * sigma * eps
eps = (z_rep - mu) / log_sigma.exp()
print(f"expected mu.grad        = {[round(v, 4) for v in (2 * z_rep).tolist()]}")
print(f"expected log_sigma.grad = {[round(v, 4) for v in (2 * z_rep * log_sigma.exp() * eps).tolist()]}")

## A tiny VAE, end to end


*Expected output starts with:* `step    0  recon 5.1227  KL 0.1519`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Data: mixture of two 2D Gaussians at (-2, 0) and (+2, 0), std 0.5
def real_batch(n):
    side = (torch.rand(n, 1) < 0.5).float() * 4 - 2      # -2 or +2
    return torch.cat([side, torch.zeros(n, 1)], dim=1) + 0.5 * torch.randn(n, 2)

enc = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 4))  # -> mu (2), logvar (2)
dec = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 2))
opt = torch.optim.Adam(list(enc.parameters()) + list(dec.parameters()), lr=1e-3)

for step in range(4000):
    x = real_batch(256)
    h = enc(x)
    mu, logvar = h[:, :2], h[:, 2:]
    z = mu + (0.5 * logvar).exp() * torch.randn_like(mu)   # reparameterization
    recon = ((dec(z) - x) ** 2).sum(dim=1).mean()          # reconstruction term
    kl = (-0.5 * (1 + logvar - mu ** 2 - logvar.exp()).sum(dim=1)).mean()
    loss = recon + 0.1 * kl                                # beta = 0.1
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 1000 == 0 or step == 3999:
        print(f"step {step:4d}  recon {recon.item():.4f}  KL {kl.item():.4f}")

# Generate: sample z from the prior N(0, I), decode
with torch.no_grad():
    gen = dec(torch.randn(4000, 2))
frac_left = ((gen[:, 0] + 2).abs() < 1).float().mean()
frac_right = ((gen[:, 0] - 2).abs() < 1).float().mean()
print(f"generated samples near left mode:  {frac_left.item():.4f}")
print(f"generated samples near right mode: {frac_right.item():.4f}")
print(f"generated samples in the gap (|x1| < 1): "
      f"{(gen[:, 0].abs() < 1).float().mean().item():.4f}")

## Turning the beta dial


*Expected output starts with:* `  beta    recon       KL  near a mode  sample std(x1)`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Same two-cluster data as the tiny VAE above
def real_batch(n):
    side = (torch.rand(n, 1) < 0.5).float() * 4 - 2
    return torch.cat([side, torch.zeros(n, 1)], dim=1) + 0.5 * torch.randn(n, 2)

def train_vae(beta, steps=4000):
    torch.manual_seed(0)
    enc = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 4))
    dec = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 2))
    opt = torch.optim.Adam(list(enc.parameters()) + list(dec.parameters()), lr=1e-3)
    for step in range(steps):
        x = real_batch(256)
        h = enc(x)
        mu, logvar = h[:, :2], h[:, 2:]
        z = mu + (0.5 * logvar).exp() * torch.randn_like(mu)
        recon = ((dec(z) - x) ** 2).sum(dim=1).mean()
        kl = (-0.5 * (1 + logvar - mu ** 2 - logvar.exp()).sum(dim=1)).mean()
        (recon + beta * kl).backward()
        opt.step(); opt.zero_grad()
    with torch.no_grad():
        gen = dec(torch.randn(4000, 2))
        near = (((gen[:, 0] + 2).abs() < 1) | ((gen[:, 0] - 2).abs() < 1)).float().mean()
        spread = gen[:, 0].std()
    return recon.item(), kl.item(), near.item(), spread.item()

print(f"{'beta':>6} {'recon':>8} {'KL':>8} {'near a mode':>12} {'sample std(x1)':>15}")
for beta in [0.01, 0.1, 1.0, 5.0, 20.0]:
    r, k, near, spread = train_vae(beta)
    print(f"{beta:>6.2f} {r:>8.4f} {k:>8.4f} {near:>12.4f} {spread:>15.4f}")
ref = real_batch(4000)
near_ref = (((ref[:, 0] + 2).abs() < 1) | ((ref[:, 0] - 2).abs() < 1)).float().mean()
print(f"reference (real data): near-a-mode {near_ref.item():.4f}  std(x1) {ref[:, 0].std().item():.4f}")

## What the latent space learns


*Expected output starts with:* `z_left  = (0.8202, 0.0062)`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

def real_batch(n):
    side = (torch.rand(n, 1) < 0.5).float() * 4 - 2
    return torch.cat([side, torch.zeros(n, 1)], dim=1) + 0.5 * torch.randn(n, 2)

# Train the same tiny VAE as above (beta = 0.1)
enc = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 4))
dec = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 2))
opt = torch.optim.Adam(list(enc.parameters()) + list(dec.parameters()), lr=1e-3)
for step in range(4000):
    x = real_batch(256)
    h = enc(x)
    mu, logvar = h[:, :2], h[:, 2:]
    z = mu + (0.5 * logvar).exp() * torch.randn_like(mu)
    recon = ((dec(z) - x) ** 2).sum(dim=1).mean()
    kl = (-0.5 * (1 + logvar - mu ** 2 - logvar.exp()).sum(dim=1)).mean()
    loss = recon + 0.1 * kl
    opt.zero_grad(); loss.backward(); opt.step()

# Encode one point from each cluster, then walk the straight line between codes
with torch.no_grad():
    x_left = torch.tensor([[-2.0, 0.0]])
    x_right = torch.tensor([[2.0, 0.0]])
    z_left = enc(x_left)[:, :2]
    z_right = enc(x_right)[:, :2]
    print(f"z_left  = ({z_left[0,0]:.4f}, {z_left[0,1]:.4f})")
    print(f"z_right = ({z_right[0,0]:.4f}, {z_right[0,1]:.4f})")
    print(f"{'alpha':>6} {'z':>20} {'decoded x':>20}")
    for a in [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]:
        z = (1 - a) * z_left + a * z_right
        xd = dec(z)
        print(f"{a:>6.1f} ({z[0,0]:>8.4f},{z[0,1]:>8.4f}) ({xd[0,0]:>8.4f},{xd[0,1]:>8.4f})")

*Expected output starts with:* `plain AE           : |corr(latent_i, factor_j)| =`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Ground-truth factors: f1 ~ N(0,1), f2 ~ N(0,0.5^2), independent.
# Observations mix them through a 45-degree rotation plus small noise.
n = 4000
f = torch.randn(n, 2) * torch.tensor([1.0, 0.5])
c, s = 0.7071, 0.7071
R = torch.tensor([[c, -s], [s, c]])
x = f @ R.T + 0.01 * torch.randn(n, 2)

def factor_corr(z):
    """|correlation| between each latent (column of z) and each true factor."""
    m = torch.corrcoef(torch.cat([z, f], dim=1).T)[:2, 2:].abs()
    return m

def report(name, z):
    m = factor_corr(z)
    print(f"{name}: |corr(latent_i, factor_j)| =")
    for i in range(2):
        print(f"    z{i+1}:  f1 {m[i,0].item():.4f}   f2 {m[i,1].item():.4f}")

# Plain autoencoder, 2D code
torch.manual_seed(0)
ae = nn.Sequential(nn.Linear(2, 32), nn.Tanh(), nn.Linear(32, 2),
                   nn.Linear(2, 32), nn.Tanh(), nn.Linear(32, 2))
enc_ae = ae[:3]
opt = torch.optim.Adam(ae.parameters(), lr=1e-2)
for step in range(3000):
    loss = ((ae(x) - x) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad():
    report("plain AE           ", enc_ae(x))

# VAE with diagonal Gaussian posterior, beta = 0.05
torch.manual_seed(0)
enc = nn.Sequential(nn.Linear(2, 32), nn.Tanh(), nn.Linear(32, 4))
dec = nn.Sequential(nn.Linear(2, 32), nn.Tanh(), nn.Linear(32, 2))
opt = torch.optim.Adam(list(enc.parameters()) + list(dec.parameters()), lr=1e-2)
for step in range(3000):
    h = enc(x)
    mu, logvar = h[:, :2], h[:, 2:]
    z = mu + (0.5 * logvar).exp() * torch.randn_like(mu)
    recon = ((dec(z) - x) ** 2).sum(dim=1).mean()
    kl = (-0.5 * (1 + logvar - mu ** 2 - logvar.exp()).sum(dim=1)).mean()
    loss = recon + 0.05 * kl
    opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad():
    report("VAE (beta = 0.05)  ", enc(x)[:, :2])

## When the latent is abandoned: posterior collapse


*Expected output starts with:* `two clusters:  final KL 1.4967   decoder sigma^2 0.1526   std of dec(z) over z 1.1276`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Two datasets. "clusters": the two-mode data from above -- real structure to encode.
# "noise": pure N(0, I) -- nothing to encode beyond what a noise model explains.
def clusters(n):
    side = (torch.rand(n, 1) < 0.5).float() * 4 - 2
    return torch.cat([side, torch.zeros(n, 1)], dim=1) + 0.5 * torch.randn(n, 2)

def noise(n):
    return torch.randn(n, 2)

def train(data_fn, steps=4000):
    torch.manual_seed(0)
    enc = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 4))
    dec = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 2))
    log_s2 = torch.zeros(1, requires_grad=True)   # learned decoder output variance
    opt = torch.optim.Adam(list(enc.parameters()) + list(dec.parameters()) + [log_s2],
                           lr=1e-3)
    for step in range(steps):
        x = data_fn(256)
        h = enc(x)
        mu, logvar = h[:, :2], h[:, 2:]
        z = mu + (0.5 * logvar).exp() * torch.randn_like(mu)
        # Gaussian NLL with learned variance s2 (up to a constant):
        # 0.5 * [ (x - x_hat)^2 / s2 + log s2 ] summed over coordinates
        nll = 0.5 * (((dec(z) - x) ** 2) / log_s2.exp() + log_s2).sum(dim=1).mean()
        kl = (-0.5 * (1 + logvar - mu ** 2 - logvar.exp()).sum(dim=1)).mean()
        loss = nll + kl
        opt.zero_grad(); loss.backward(); opt.step()
    # How much does the decoder's output actually depend on z?
    with torch.no_grad():
        zs = torch.randn(1000, 2)
        dec_spread = dec(zs).std(dim=0).mean()
    return kl.item(), log_s2.exp().item(), dec_spread.item()

for name, fn in [("two clusters", clusters), ("pure noise  ", noise)]:
    kl, s2, spread = train(fn)
    print(f"{name}:  final KL {kl:.4f}   decoder sigma^2 {s2:.4f}   "
          f"std of dec(z) over z {spread:.4f}")

## Try it yourself

1. In the arc autoencoder, widen the bottleneck from 1 to 2 latent dimensions. Predict what happens to the linear model's MSE before you run it (the arc lives in 2D — what is the best a 2D linear map can do?), then verify.
2. Rerun the tiny VAE with `beta` set to 0.001, 0.1, and 1.0. For each, record the final reconstruction and KL values and the fraction of prior samples in the gap. Which `beta` gives the best trade-off between mode coverage and gap samples?
3. Implement a latent traversal for the trained VAE: decode `z = (a, 0)` for `a` from -3 to 3 in steps of 0.5 and print the outputs. Where in latent space does the decoder switch from the left cluster to the right?
4. Replace `rsample()`-style sampling in the tiny VAE with `mu` alone (no noise at all). Training still runs — but what happened to the KL term's meaning, and what do prior samples look like now?
5. In the disentanglement experiment, change the rotation angle from 45 to 10 degrees and equalize the two factor scales to 1.0. Rerun both models. Does the VAE still recover the factors? Connect what you observe to the Locatello et al. impossibility argument.
6. In the posterior-collapse experiment, implement KL annealing: multiply the KL term by `min(1, step / 2000)`. Does the two-cluster model reach a different final KL? Then try the noise dataset — can annealing rescue a latent that has nothing to encode?


---

Full discussion of everything above: [S23 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s23/).
